In [0]:
%run ./00_Setup

In [0]:
# ============================================
# BRONZE PATHS
# ============================================

BRONZE_SCHEMA = f"{CATALOG}.bronze"

BRONZE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers"
BRONZE_BOOKS = f"{BRONZE_SCHEMA}.books"
BRONZE_ORDERS = f"{BRONZE_SCHEMA}.orders"

print("Bronze Customers:", BRONZE_CUSTOMERS)
print("Bronze Books    :", BRONZE_BOOKS)
print("Bronze Orders   :", BRONZE_ORDERS)

In [0]:
# ============================================
# CREATE BRONZE SCHEMA
# ============================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}
""")

print("Bronze schema is ready.")

In [0]:
# ============================================
# AUTO LOADER - CUSTOMERS CSV
# ============================================

customers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option(
        "cloudFiles.schemaLocation",
        f"{BASE_PATH}/schemas/customers"
    )
    .option("header", "true")
    .load(CUSTOMERS_PATH)
)

print("Customers Auto Loader stream created successfully.")

In [0]:
# ============================================
# WRITE CUSTOMERS TO BRONZE
# ============================================

customers_query = (
    customers_stream
    .writeStream
    .queryName("customers_bronze_autoloader")
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{BASE_PATH}/checkpoints/customers"
    )
    #.trigger(processingTime="10 seconds")
    .trigger(availableNow=True)
    .toTable(BRONZE_CUSTOMERS)
)

In [0]:
# ============================================
# 06. AUTO LOADER - BOOKS JSON
# ============================================

books_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        f"{BASE_PATH}/schemas/books"
    )
    .load(BOOKS_PATH)
)

print("Books Auto Loader stream created.")

In [0]:
# ============================================
# 07. BOOKS → BRONZE
# ============================================

books_query = (
    books_stream
    .writeStream
    .queryName("books_bronze_autoloader")
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{BASE_PATH}/checkpoints/books"
    )
    #.trigger(processingTime="10 seconds")
    .trigger(availableNow=True)
    .toTable(BRONZE_BOOKS)
)

In [0]:
# ============================================
# 08. AUTO LOADER - ORDERS PARQUET
# ============================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        f"{BASE_PATH}/schemas/orders"
    )
    .load(ORDERS_PATH)
)

print("Orders Auto Loader stream created.")

In [0]:
# ============================================
# 09. ORDERS → BRONZE
# ============================================

orders_query = (
    orders_stream
    .writeStream
    .queryName("orders_bronze_autoloader")
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{BASE_PATH}/checkpoints/orders"
    )
    #.trigger(processingTime="10 seconds")
    .trigger(availableNow=True)
    .toTable(BRONZE_ORDERS)
)

In [0]:
# ============================================
# 10. CHECK ACTIVE STREAMING QUERIES
# ============================================

for query in spark.streams.active:

    print(
        f"Name   : {query.name}"
    )

    print(
        f"Active : {query.isActive}"
    )

    print("-" * 50)

In [0]:
# ============================================
# 11. VIEW BRONZE CUSTOMERS
# ============================================

display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_CUSTOMERS}
        ORDER BY created_at DESC
    """)
)

In [0]:
# ============================================
# 12. VIEW BRONZE BOOKS
# ============================================

display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_BOOKS}
        ORDER BY created_at DESC
    """)
)

In [0]:
# ============================================
# 13. VIEW BRONZE ORDERS
# ============================================

display(
    spark.sql(f"""
        SELECT *
        FROM {BRONZE_ORDERS}
        ORDER BY order_timestamp DESC
    """)
)

In [0]:
# ============================================
# 14. CHECK BRONZE RECORD COUNTS
# ============================================

print("Customers:")
display(
    spark.sql(f"""
        SELECT COUNT(*) AS record_count
        FROM {BRONZE_CUSTOMERS}
    """)
)

print("Books:")
display(
    spark.sql(f"""
        SELECT COUNT(*) AS record_count
        FROM {BRONZE_BOOKS}
    """)
)

print("Orders:")
display(
    spark.sql(f"""
        SELECT COUNT(*) AS record_count
        FROM {BRONZE_ORDERS}
    """)
)

In [0]:
# ============================================
# 15. CHECK SOURCE FILES
# ============================================

print("CUSTOMER FILES")
display(dbutils.fs.ls(CUSTOMERS_PATH))

print("BOOK FILES")
display(dbutils.fs.ls(BOOKS_PATH))

print("ORDER FILES")
display(dbutils.fs.ls(ORDERS_PATH))

In [0]:
# ============================================
# 16. STOP ALL STREAMING QUERIES
# ============================================

for query in spark.streams.active:

    query.stop()

print("All streaming queries stopped.")